#BRONZE LAYER

In [0]:
# Bronze Layer - raw tables
bronze_drivers = spark.table("workspace.default.drivers")
bronze_races   = spark.table("workspace.default.races")
bronze_results = spark.table("workspace.default.results")


#SILVER LAYER

In [0]:
from pyspark.sql.functions import col, concat_ws, expr

# Clean Drivers
silver_drivers = bronze_drivers.select(
    expr("try_cast(driverId as int)").alias("driverId"),
    concat_ws(" ", col("forename"), col("surname")).alias("driver_name"),
    expr("try_cast(dob as date)").alias("dob"),
    col("nationality")
)

# Clean Races
silver_races = bronze_races.select(
    expr("try_cast(raceId as int)").alias("raceId"),
    expr("try_cast(year as int)").alias("year"),
    expr("try_cast(round as int)").alias("round"),
    col("name").alias("race_name"),
    expr("try_cast(date as date)").alias("date")
)

# Clean Results
silver_results = bronze_results.select(
    expr("try_cast(resultId as int)").alias("resultId"),
    expr("try_cast(raceId as int)").alias("raceId"),
    expr("try_cast(driverId as int)").alias("driverId"),
    expr("try_cast(constructorId as int)").alias("constructorId"),
    expr("try_cast(grid as int)").alias("grid"),
    expr("try_cast(positionOrder as int)").alias("positionOrder"),
    expr("try_cast(points as double)").alias("points"),
    expr("try_cast(laps as int)").alias("laps"),
    expr("try_cast(milliseconds as long)").alias("milliseconds"),
    col("fastestLapTime"),
    col("fastestLapSpeed"),
    expr("try_cast(statusId as int)").alias("statusId")
)

# Save silver tables as delta
silver_drivers.write.format("delta").mode("overwrite").saveAsTable("silver_drivers")
silver_races.write.format("delta").mode("overwrite").saveAsTable("silver_races")
silver_results.write.format("delta").mode("overwrite").saveAsTable("silver_results")

#GOLD LAYER

In [0]:
# Load Silver
drivers = spark.table("silver_drivers")
races   = spark.table("silver_races")
results = spark.table("silver_results")

# Join Results + Races + Drivers
joined = results.join(drivers, "driverId") \
                .join(races, "raceId")

# Example 1: Total Points by Driver & Year
gold_points = joined.groupBy("year", "driver_name") \
    .sum("points") \
    .withColumnRenamed("sum(points)", "total_points")

gold_points.write.format("delta").mode("overwrite").saveAsTable("gold_driver_points")

# Example 2: Wins (positionOrder = 1)
gold_wins = joined.filter(col("positionOrder") == 1) \
    .groupBy("year", "driver_name") \
    .count() \
    .withColumnRenamed("count", "wins")

gold_wins.write.format("delta").mode("overwrite").saveAsTable("gold_driver_wins")

# Example 3: Average Laps Completed per Driver
gold_avg_laps = joined.groupBy("driver_name") \
    .avg("laps") \
    .withColumnRenamed("avg(laps)", "avg_laps")

gold_avg_laps.write.format("delta").mode("overwrite").saveAsTable("gold_driver_avg_laps")


In [0]:
display(gold_points.filter("year=2021"))

Databricks visualization. Run in Databricks to view.

In [0]:
display(gold_wins.filter("year = 2020"))
